First of all, we init the SPARQLWrapper service with the SPARQL endpoint

In [ ]:
!pip install SPARQLWrapper

In [1]:
import csv
from SPARQLWrapper import SPARQLWrapper

sparql = SPARQLWrapper("https://query.wikidata.org/sparql")

Then we define our CONSTRUCT query to extract the metadata

In [6]:
sparql.setQuery("""
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX wd: <http://www.wikidata.org/entity/>

CONSTRUCT {
  wd:Q275929 wdt:P31 ?type .
  wd:Q275929 wdt:P18 ?image .
  wd:Q275929 wdt:P27 ?citizenship .
  wd:Q275929 wdt:P569 ?dateBirth .
  ?citizenship rdfs:label ?citizenshipLabel .
  wd:Q275929 wdt:P19 ?placeBirth .
  ?placeBirth rdfs:label ?placeBirthLabel .
  wd:Q275929 wdt:P19 ?placeDeath .
  ?placeDeath rdfs:label ?placeDeathLabel .
  wd:Q275929 wdt:P119 ?burial .
  ?burial rdfs:label ?burialLabel .
  wd:Q275929 wdt:P1442 ?graveImage .
  wd:Q275929 wdt:P103 ?language.
  ?language rdfs:label ?languageLabel .
  wd:Q275929 wdt:P106 ?occupation.
  ?occupation rdfs:label ?occupationLabel .
  wd:Q275929 wdt:P135 ?movement.
  ?movement rdfs:label ?movementLabel .
  wd:Q275929 wdt:P800 ?work.
  ?work rdfs:label ?workLabel .
}
WHERE {
  wd:Q275929 wdt:P31 ?type .
  wd:Q275929 wdt:P18 ?image .
  wd:Q275929 wdt:P27 ?citizenship .
  wd:Q275929 wdt:P569 ?dateBirth .     
  wd:Q275929 wdt:P19 ?placeBirth .
  ?citizenship rdfs:label ?citizenshipLabel  FILTER (lang(?citizenshipLabel) = 'en') .
  ?placeBirth rdfs:label ?placeBirthLabel  FILTER (lang(?placeBirthLabel) = 'en') .
  wd:Q275929 wdt:P570 ?dateDeath .     
  wd:Q275929 wdt:P20 ?placeDeath .
  ?placeDeath rdfs:label ?placeDeathLabel  FILTER (lang(?placeDeathLabel) = 'en') .
  wd:Q275929 wdt:P509 ?causeDeath .
  ?causeDeath rdfs:label ?causeDeathLabel  FILTER (lang(?causeDeathLabel) = 'en') .
  wd:Q275929 wdt:P119 ?burial .
  ?burial rdfs:label ?burialLabel  FILTER (lang(?burialLabel) = 'en') .
  OPTIONAL {wd:Q275929 wdt:P1442 ?graveImage .}
  wd:Q275929 wdt:P103 ?language .
  ?language rdfs:label ?languageLabel  FILTER (lang(?languageLabel) = 'en') .
  wd:Q275929 wdt:P106 ?occupation .
  ?occupation rdfs:label ?occupationLabel  FILTER (lang(?occupationLabel) = 'en') .
  wd:Q275929 wdt:P135 ?movement .
  ?movement rdfs:label ?movementLabel  FILTER (lang(?movementLabel) = 'es') .
  wd:Q275929 wdt:P800 ?work .
  ?work rdfs:label ?workLabel  FILTER (lang(?workLabel) = 'es') .
}
"""
)

Finally, we serialise the result

In [7]:
results = sparql.queryAndConvert()
print(results.serialize())

@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix wd: <http://www.wikidata.org/entity/> .
@prefix wdt: <http://www.wikidata.org/prop/direct/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

wd:Q275929 wdt:P103 wd:Q1321 ;
    wdt:P106 wd:Q113516626,
        wd:Q11774202,
        wd:Q15949613,
        wd:Q1607826,
        wd:Q1930187,
        wd:Q3068305,
        wd:Q36180,
        wd:Q4263842,
        wd:Q49757,
        wd:Q6625963 ;
    wdt:P119 wd:Q16746788,
        wd:Q6116696 ;
    wdt:P135 wd:Q667661 ;
    wdt:P18 <http://commons.wikimedia.org/wiki/Special:FilePath/Emilia%20Pardo%20Baz%C3%A1n%20por%20Luis%20Sellier%201885.jpg> ;
    wdt:P19 wd:Q2807,
        wd:Q8757 ;
    wdt:P27 wd:Q29 ;
    wdt:P31 wd:Q5 ;
    wdt:P569 "1851-09-16T00:00:00+00:00"^^xsd:dateTime ;
    wdt:P800 wd:Q12391421,
        wd:Q16578932,
        wd:Q16588120,
        wd:Q16588138,
        wd:Q16606269,
        wd:Q19437333,
        wd:Q5575234,
        wd:Q9024322 .

wd:Q113516626 rdfs:

In [8]:
with open("output/emilia-pardo-bazan.ttl", "w") as text_file:
    text_file.write(results.serialize())

#### We can also run a federated SPARQL query to retrieve integrated data from external repositories such as the National Library of France.

In [10]:
query = """
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX rdarelationships: <http://rdvocab.info/RDARelationshipsWEMI/>
PREFIX rdagroup1elements: <http://rdvocab.info/Elements/>

SELECT DISTINCT ?author ?expression ?title ?edition ?placeOfPublication ?yearOfPublication ?langCode WHERE {
wd:Q275929 wdt:P268 ?id
BIND(uri(concat(concat("http://data.bnf.fr/ark:/12148/cb", ?id),"#about")) as ?author)
SERVICE <http://data.bnf.fr/sparql> {
  ?expression <http://id.loc.gov/vocabulary/relators/aut> ?author .
  OPTIONAL {?expression dcterms:language ?langCode .}
  OPTIONAL {?manifestation dcterms:publisher ?edition .}
  ?manifestation rdarelationships:expressionManifested ?expression .
  ?manifestation dcterms:title ?title .
  ?manifestation dcterms:date ?yearOfPublication .
  OPTIONAL{ ?manifestation rdagroup1elements:placeOfPublication ?placeOfPublication .}
  }
}
LIMIT 5000
"""

In [11]:
from SPARQLWrapper import SPARQLWrapper, JSON

sparql.setReturnFormat(JSON)

sparql.setQuery(query)

In [12]:
try:
    ret = sparql.queryAndConvert()

    for r in ret["results"]["bindings"]:
        print(r)
except Exception as e:
    print(e)

{'author': {'type': 'uri', 'value': 'http://data.bnf.fr/ark:/12148/cb12004268k#about'}, 'yearOfPublication': {'type': 'literal', 'value': '2021'}, 'expression': {'type': 'uri', 'value': 'http://data.bnf.fr/ark:/12148/cb47194881g#Expression'}, 'placeOfPublication': {'type': 'literal', 'value': 'Cádiz'}, 'langCode': {'type': 'uri', 'value': 'http://id.loc.gov/vocabulary/iso639-2/spa'}, 'edition': {'type': 'literal', 'value': 'Cádiz : Cazador de ratas editorial , 2021'}, 'title': {'type': 'literal', 'value': 'La última fada'}}
{'author': {'type': 'uri', 'value': 'http://data.bnf.fr/ark:/12148/cb12004268k#about'}, 'yearOfPublication': {'type': 'literal', 'value': '2021'}, 'expression': {'type': 'uri', 'value': 'http://data.bnf.fr/ark:/12148/cb474010549#Expression'}, 'placeOfPublication': {'type': 'literal', 'value': 'Cádiz'}, 'langCode': {'type': 'uri', 'value': 'http://id.loc.gov/vocabulary/iso639-2/spa'}, 'edition': {'type': 'literal', 'value': 'Cádiz : Cazador de ratas editorial , 2021-

Now we can create a CSV file with the content

In [13]:
output_file="output/emilia-pardo-bazan-works.csv"
with open(output_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["author", "expression", "title", "edition", "placeOfPublication", "yearOfPublication", "langCode"])

    for item in ret["results"]["bindings"]:
        author = item.get("author", {}).get("value", "")
        expression = item.get("expression", {}).get("value", "")
        title = item.get("title", {}).get("value", "")
        edition = item.get("edition", {}).get("value", "")
        placeOfPublication = item.get("placeOfPublication", {}).get("value", "")
        yearOfPublication = item.get("yearOfPublication", {}).get("value", "")
        langCode = item.get("langCode", {}).get("value", "")

        writer.writerow([author, expression, title, edition, placeOfPublication, yearOfPublication, langCode])

#### Now we can analyse the content retrieved

In [14]:
import pandas as pd

# Load the CSV
df = pd.read_csv(output_file)

# Show first rows
print("First rows:")
print(df.head())

# Convert publication date to datetime
df["yearOfPublication"] = pd.to_datetime(df["yearOfPublication"], errors="coerce")

# Extract year
df["Year"] = df["yearOfPublication"].dt.year

# Basic info
print("\nData info:")
print(df.info())

# Count works per year
works_per_year = df["Year"].value_counts().sort_index()

print("\nWorks per year:")
print(works_per_year)

# Find earliest and latest works
earliest = df.loc[df["Year"].idxmin()]
latest = df.loc[df["Year"].idxmax()]

print("\nEarliest work:")
print(earliest)

print("\nLatest work:")
print(latest)

# Save summary to new CSV
works_per_year.to_csv("output/emilia-pardo-bazan-works_per_year.csv")

First rows:
                                            author  \
0  http://data.bnf.fr/ark:/12148/cb12004268k#about   
1  http://data.bnf.fr/ark:/12148/cb12004268k#about   
2  http://data.bnf.fr/ark:/12148/cb12004268k#about   
3  http://data.bnf.fr/ark:/12148/cb12004268k#about   
4  http://data.bnf.fr/ark:/12148/cb12004268k#about   

                                          expression                  title  \
0  http://data.bnf.fr/ark:/12148/cb47194881g#Expr...         La última fada   
1  http://data.bnf.fr/ark:/12148/cb474010549#Expr...  Colección Pardo Bazán   
2  http://data.bnf.fr/ark:/12148/cb44515845g#Expr...        Cuentos de amor   
3  http://data.bnf.fr/ark:/12148/cb44274657d#Expr...     Le château d'Ulloa   
4  http://data.bnf.fr/ark:/12148/cb35306266h#Expr...     Le Château d'Ulloa   

                                      edition placeOfPublication  \
0   Cádiz : Cazador de ratas editorial , 2021              Cádiz   
1  Cádiz : Cazador de ratas editorial , 2021-       

#### Now get all the distinct places of publication

In [15]:
unique_values = df["placeOfPublication"].unique()
print(unique_values)

['Cádiz' 'Madrid' '[Paris]' 'Barcelona. - Bogotá. - Buenos Aires [et al.]'
 '[Madrid]' 'Pessac' 'Vigo' 'Barcelona' 'La Coruña'
 'Valencina de la Concepción (Sevilla)' '[Le Vésinet]' '[S. l.]'
 '[Huesca (Espagne)]' 'Madrid. - Frankfurt am Main' 'Exeter (GB)'
 'Boston. - New York. - Chicago' 'Vigo (España)'
 'Madrid. - [La Coruña, Spain]' 'Sevilla' 'Alicante' 'León' 'Paris'
 'Buenos Aires' 'Nantes' 'Granada' 'Santiago de Compostela']


#### Now get all the languages

In [16]:
unique_values = df["langCode"].unique()
print(unique_values)

['http://id.loc.gov/vocabulary/iso639-2/spa'
 'http://id.loc.gov/vocabulary/iso639-2/fre' nan]


In [17]:
# Count works per language
works_per_language = df["langCode"].value_counts().sort_index()

print("\nWorks per language:")
print(works_per_language)


Works per language:
langCode
http://id.loc.gov/vocabulary/iso639-2/fre    16
http://id.loc.gov/vocabulary/iso639-2/spa    73
Name: count, dtype: int64


#### References
- Candela, G. (2023). Towards a semantic approach in GLAM Labs: The case of the Data Foundry at the National Library of Scotland. Journal of Information Science, 52(1), 3–21. https://doi.org/10.1177/01655515231174386
- Dişli, M., Osti, G., Candela, G., & Zijdeman, R. (2025). From Linked Open Data to Collections as Data: A Reproducible Framework Using Federated Queries. Information Technology and Libraries, 44(4). https://doi.org/10.5860/ital.v44i4.17432